# sheet_operate — Phase 3 SFT（LoRA）
Qwen3-4B-Instruct-2507 ＋ LoRA r=64 ＋ 通用資料混合（抗遺忘）＋ Drive checkpoint 斷點續訓

**使用步驟**
1. Colab 選 GPU 執行階段（96GB 或 A100/H100 皆可，4B LoRA 需求不高）
2. 填好下方 `CFG`（repo_url 或先把 repo 放到 Drive）
3. 全部執行。session 斷線後重跑全部 cell 會自動從最後一個 checkpoint 續訓
4. 流程：基準線 eval（裸模型）→ SFT → 訓後 eval → 通用能力抽查


In [ ]:
# ===== 設定 =====
CFG = dict(
    repo_url   = "",   # 例 "https://github.com/<user>/sheet_operate.git"；private repo 用 https://<token>@github.com/... 形式
    drive_root = "/content/drive/MyDrive/sheet_operate",   # checkpoint 與輸出存放處
    base_model = "Qwen/Qwen3-4B-Instruct-2507",
    max_seq_len = 4096,
    lora_r = 64, lora_alpha = 64,
    lr = 2e-4, epochs = 2,
    per_device_bs = 8, grad_accum = 2,      # 有效 batch 16
    mix_general = True, general_n = 110,     # 通用中文指令資料混合量（約 15%）
    run_baseline = True,                     # 訓練前先量裸模型基準線
    eval_limit = None,                       # 想快速煙霧測試可設 24
    seed = 3407,
)

In [ ]:
%%capture
# ===== 安裝依賴 =====
!pip install unsloth
!pip install openpyxl

In [ ]:
# ===== 掛載 Drive、取得 repo、重生評測任務 =====
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CFG["drive_root"], exist_ok=True)
REPO = "/content/sheet_operate"

if CFG["repo_url"]:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", CFG["repo_url"], REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    # 沒填 repo_url 時，改用事先放在 Drive 的 repo 副本
    REPO = os.path.join(CFG["drive_root"], "repo")
    assert os.path.exists(REPO), "請填 CFG['repo_url']，或把 repo 放到 Drive 的 sheet_operate/repo"

sys.path.insert(0, REPO)
os.chdir(REPO)

# 評測任務由 seed 確定性重生（與本機一致）
subprocess.run([sys.executable, "scripts/gen_tasks.py", "--out", "data/tasks/eval",
                "--families", "all", "--n", "8", "--seed", "900001"], check=True)
CKPT_DIR = os.path.join(CFG["drive_root"], "ckpt_sft_lora")
print("repo:", REPO)
print("checkpoints:", CKPT_DIR)

In [ ]:
# ===== 載入基礎模型 =====
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["base_model"],
    max_seq_length = CFG["max_seq_len"],
    dtype          = None,          # 自動（bf16）
    load_in_4bit   = False,         # 96GB 跑 bf16 綽綽有餘
)

In [ ]:
# ===== Gym 評測函式（基準線與訓後共用；逐題：生成→沙盒執行→逐格驗證） =====
import json as _json
from pathlib import Path
from sheetops.encoder import encode_workbook
from sheetops.env import solve_once
from sheetops.executor import extract_code
from sheetops.prompts import SYSTEM_PROMPT, build_user_prompt

def run_gym_eval(model, tokenizer, tasks_dir="data/tasks/eval", limit=None, tag=""):
    FastLanguageModel.for_inference(model)
    task_dirs = sorted(p.parent for p in Path(tasks_dir).glob("*/task.json"))
    if limit:
        task_dirs = task_dirs[:limit]
    rows = []
    for i, td in enumerate(task_dirs):
        spec = _json.loads((td / "task.json").read_text(encoding="utf-8"))
        user = build_user_prompt(spec["instruction"],
                                 encode_workbook(td / "start.xlsx"),
                                 spec.get("context", ""))
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
        code_ = extract_code(reply)
        if code_:
            rep = solve_once(td, code_)
            ok, score = bool(rep["full_match"]), rep["score"]
        else:
            ok, score = False, 0.0
        rows.append({"id": spec["id"], "family": spec["family"], "pass": ok, "score": score})
        if (i + 1) % 12 == 0:
            print(f"  [{tag}] {i+1}/{len(task_dirs)}  pass so far: {sum(r['pass'] for r in rows)}")
    import pandas as pd
    df = pd.DataFrame(rows)
    print(f"\n[{tag}] overall pass@1 = {df['pass'].mean():.1%}  (avg score {df['score'].mean():.3f})")
    print(df.groupby("family")["pass"].mean().sort_values().to_string())
    return df

In [ ]:
# ===== （建議開啟）基準線：裸 Qwen3-4B-Instruct 的 pass@1 =====
baseline_df = None
if CFG["run_baseline"]:
    baseline_df = run_gym_eval(model, tokenizer, limit=CFG["eval_limit"], tag="baseline")
    baseline_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline.csv"), index=False)

In [ ]:
# ===== 掛上 LoRA =====
model = FastLanguageModel.get_peft_model(
    model,
    r = CFG["lora_r"],
    lora_alpha = CFG["lora_alpha"],
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = CFG["seed"],
)

In [ ]:
# ===== 組訓練資料：蒸餾軌跡 ＋ 通用中文指令混合（抗遺忘） =====
import random
from datasets import Dataset

random.seed(CFG["seed"])
samples = []
with open("data/sft/teacher_sft.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = _json.loads(line)
        samples.append(rec["messages"])
print(f"蒸餾軌跡：{len(samples)} 條")

if CFG["mix_general"]:
    try:
        from datasets import load_dataset
        gen_ds = load_dataset("yentinglin/TaiwanChat", split="train", streaming=True)
        got = 0
        for ex in gen_ds:
            msgs = ex.get("messages") or []
            msgs = [m for m in msgs if m.get("role") in ("user", "assistant")]
            if len(msgs) >= 2 and msgs[0]["role"] == "user":
                samples.append(msgs[:2])
                got += 1
            if got >= CFG["general_n"]:
                break
        print(f"通用資料混入：{got} 條（TaiwanChat）")
    except Exception as e:
        print(f"[警告] 通用資料載入失敗，僅用任務資料續訓：{e}")

random.shuffle(samples)
texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in samples]
train_ds = Dataset.from_dict({"text": texts})
print(f"訓練樣本總數：{len(train_ds)}")

In [ ]:
# ===== 訓練（只對 assistant 回覆算 loss；自動續訓） =====
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from transformers.trainer_utils import get_last_checkpoint

os.makedirs(CKPT_DIR, exist_ok=True)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = CFG["max_seq_len"],
        output_dir = CKPT_DIR,
        per_device_train_batch_size = CFG["per_device_bs"],
        gradient_accumulation_steps = CFG["grad_accum"],
        num_train_epochs = CFG["epochs"],
        learning_rate = CFG["lr"],
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.03,
        weight_decay = 0.01,
        logging_steps = 10,
        save_steps = 50,
        save_total_limit = 3,
        bf16 = True,
        seed = CFG["seed"],
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
last_ckpt = get_last_checkpoint(CKPT_DIR)
print("resume from:", last_ckpt)
trainer.train(resume_from_checkpoint = last_ckpt)

In [ ]:
# ===== 存 LoRA adapter 到 Drive =====
ADAPTER_DIR = os.path.join(CFG["drive_root"], "adapter_sft_v1")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to:", ADAPTER_DIR)
# Phase 5 部署時再 merge：
# model.save_pretrained_merged(os.path.join(CFG["drive_root"], "merged_sft_v1"),
#                              tokenizer, save_method="merged_16bit")

In [ ]:
# ===== 訓後評測＋與基準線比較 =====
sft_df = run_gym_eval(model, tokenizer, limit=CFG["eval_limit"], tag="sft")
sft_df.to_csv(os.path.join(CFG["drive_root"], "eval_sft_v1.csv"), index=False)

if baseline_df is not None:
    import pandas as pd
    cmp = pd.DataFrame({
        "baseline": baseline_df.groupby("family")["pass"].mean(),
        "sft":      sft_df.groupby("family")["pass"].mean(),
    })
    cmp["gain"] = cmp["sft"] - cmp["baseline"]
    print(cmp.sort_values("gain").to_string())
    print(f"\noverall: baseline {baseline_df['pass'].mean():.1%} -> sft {sft_df['pass'].mean():.1%}")

In [ ]:
# ===== 通用能力抽查（遺忘煙霧偵測器，人工看輸出是否還正常） =====
FastLanguageModel.for_inference(model)
for q in ["用兩三句話介紹台北 101。",
          "解釋什麼是複利，並舉一個簡單的例子。",
          "寫一個 Python 函式判斷質數。",
          "客戶來信抱怨出貨延遲，幫我擬一段得體的道歉回覆。"]:
    p = tokenizer.apply_chat_template([{"role": "user", "content": q}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    print("Q:", q)
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)[:400])
    print("-" * 60)

## 下一步
- pass@1 若明顯超過基準線且 composite/context_rule 有 20%+ 成功率 → 進 Phase 4 GRPO（同一個 Gym 當 reward）
- 若過擬合跡象（train loss 極低但 eval 不動）→ 換前一個 checkpoint 或減 epoch
- 部署：merge → GGUF → 本地 Ollama（Phase 5）
